# 13 — 4모델 동일 가중치 앙상블 (예측 재사용 전용)

**학습 0회 / GPU 불필요 / 12번 수정 불필요.** 10번 기존 g3 + 12번 XGB·LR50·신규 g3를 25%씩 평균합니다.

- 전체 Train OOF(6201행), Test(2546행) 파일을 기존 `experiments/preds/`에서 읽습니다.
- 기존 파일의 클래스 순서·형상·확률값·출처 메타를 검사합니다. 입력 누락 시 **재학습하지 않고 명확하게 중단**합니다.
- **전체 OOF와 개발 CV/고정 홀드아웃은 다른 검증입니다.** 전체 OOF로 고정 홀드아웃 성능을 대체하지 않습니다. 개발·홀드아웃용 *독립 예측 파일*이 없으면 해당 지표는 미검증으로 표시합니다.
- 팀장님 공유 수치 `0.5293 → 0.5365`는 참고 목표이며 직접 계산해 재현 여부를 출력합니다.
- Test CSV는 검증 확인 후 수동 허용해야 생성됩니다. 기본적으로 PR용 `.npy`와 메타만 생성합니다.

In [1]:
from pathlib import Path
import json, hashlib, os, tempfile, sys
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'src'/'config.py').exists() and (p/'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 루트(src/config.py 및 data 폴더)를 찾지 못했습니다. 프로젝트 내부에서 실행하세요.')
# 12번 노트북과 동일하게 프로젝트 루트를 import 경로에 등록
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert (ROOT/'src'/'config.py').is_file(), 'src/config.py 경로 확인 필요'
PRED = ROOT/'experiments'/'preds'
OUT = PRED
OUT_STEM = 'A_model__13_four_model_equal_xgb_lr50_oldg3_newg3'
SAVE_ENSEMBLE = True
SAVE_SUBMISSION = False  # 최종 제출 승인 전 변경 금지
ALLOW_UNVERIFIED_SUBMISSION = False  # 개발 CV·홀드아웃 예측이 없으면 제출 파일 생성 불가
print('프로젝트:',ROOT,'\n예측 폴더:',PRED)

프로젝트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype 
예측 폴더: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds


In [2]:
from src.config import DATA_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds
train,test=load_data(DATA_DIR)
X_raw,y_raw=split_xy(train)
y,le=encode_target(y_raw)
y=np.asarray(y,dtype=int)
N,NT,K=len(y),len(test),len(le.classes_)
assert (N,NT,K)==(6201,2546,26),f'데이터 변경 확인 필요: {(N,NT,K)}'
CLASSES=list(map(str,le.classes_))
print('Train/Test/Classes:',N,NT,K)

Train/Test/Classes: 6201 2546 26


## 1. 기존 저장 파일 자동 검색

12번에서 실제 저장하는 이름과 10번에서 실제 저장하는 이름을 기준으로 합니다. **모델을 새로 학습하지 않습니다.** 이름이 다른 경우 아래 `STEMS`의 해당 값만 실제 파일명에 맞게 수정하세요.

In [3]:
STEMS = {
 'xgb': 'A_model__12_weighted_xgb_E_cpu',
 'lr50': 'A_model__12_lr_position_bin50',
 'old_g3': 'A_model__lgbm_g3_hotspot3_bin_exact_1110',
 'new_g3': 'A_model__12_g3_multihot_signature',
}
# 10번 g3 저장 파일의 실제 이름. 이전 실험에서 다른 stem을 썼다면 파일 검색 결과로 수정.
for key,stem in STEMS.items():
    print(f'{key:8s}',stem, '|',*(('OOF:'+str((PRED/f'{stem}__oof.npy').exists()),'Test:'+str((PRED/f'{stem}__test.npy').exists()))))
print('\n현재 보유 g3 후보:')
for p in sorted(PRED.glob('*g3*__oof.npy')):
    print(' ',p.name)

xgb      A_model__12_weighted_xgb_E_cpu | OOF:True Test:True
lr50     A_model__12_lr_position_bin50 | OOF:True Test:True
old_g3   A_model__lgbm_g3_hotspot3_bin_exact_1110 | OOF:True Test:True
new_g3   A_model__12_g3_multihot_signature | OOF:True Test:True

현재 보유 g3 후보:
  A_model__12_ensemble_g3_signature_lr50__oof.npy
  A_model__12_g3_multihot_signature__oof.npy
  A_model__ensemble_E_xgb_position_lr_lgbm_g3_equal_1110__oof.npy
  A_model__lgbm_g3_hotspot3_bin_exact_1110__oof.npy
  A_model__lgbm_g3_recomputed_for_B_11__oof.npy
  A_model__lr_E_position_g3_3ensemble_1110__oof.npy


In [4]:
def load_prob(stem,kind,rows,require_meta=True):
    path=PRED/f'{stem}__{kind}.npy'
    if not path.exists():
        raise FileNotFoundError(f'필수 예측 파일 없음: {path}\n이 실험은 자동 재학습하지 않습니다. 해당 예측 파일을 먼저 확보해야 합니다.')
    arr=np.load(path,allow_pickle=False)
    if arr.shape!=(rows,K):
        raise ValueError(f'{path.name}: shape {arr.shape}, 기대 {(rows,K)}')
    if not np.issubdtype(arr.dtype,np.number) or not np.isfinite(arr).all():
        raise ValueError(f'{path.name}: 유효하지 않은 확률값')
    if (arr < -1e-6).any() or (arr > 1+1e-6).any() or not np.allclose(arr.sum(axis=1),1,atol=2e-3):
        raise ValueError(f'{path.name}: 확률 범위/합 불일치')
    meta_path=PRED/f'{stem}__meta.json'
    if require_meta and not meta_path.exists():
        raise FileNotFoundError(f'클래스 및 출처 확인용 메타 파일 없음: {meta_path}')
    meta=json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    if meta:
        if list(map(str,meta.get('classes',[])))!=CLASSES:
            raise ValueError(f'{stem}: 클래스 순서 불일치 또는 메타 classes 누락')
        if meta.get('train_rows',N)!=N or meta.get('test_rows',NT)!=NT:
            raise ValueError(f'{stem}: train/test 행 수 메타 불일치')
    return np.asarray(arr,dtype=np.float64)

missing=[str(PRED/f'{stem}__{kind}.npy') for stem in STEMS.values() for kind in ('oof','test') if not (PRED/f'{stem}__{kind}.npy').exists()]
if missing:
    print('\n필수 파일 누락:')
    print('\n'.join(missing))
    raise FileNotFoundError('8개 OOF/Test 예측 파일이 모두 있어야 합니다. 기존 실험 파일을 확보하세요. 학습은 실행하지 않았습니다.')

oof={name:load_prob(stem,'oof',N) for name,stem in STEMS.items()}
test_p={name:load_prob(stem,'test',NT) for name,stem in STEMS.items()}
print('4개 모델 OOF/Test 로드 및 확률/클래스 검증 완료')

4개 모델 OOF/Test 로드 및 확률/클래스 검증 완료


## 2. 전체 OOF 3모델 vs 4모델 (재학습 없음)

동일 Group 5-Fold의 Fold별 점수도 **이미 저장된 OOF를 분할해서 평가**합니다. `get_folds()`는 분할 인덱스만 반환하며 학습하지 않습니다.

In [5]:
p3_oof=(oof['xgb']+oof['lr50']+oof['old_g3'])/3
p4_oof=(oof['xgb']+oof['lr50']+oof['old_g3']+oof['new_g3'])/4
p3_test=(test_p['xgb']+test_p['lr50']+test_p['old_g3'])/3
p4_test=(test_p['xgb']+test_p['lr50']+test_p['old_g3']+test_p['new_g3'])/4

def f1(a,p):return float(f1_score(a,p.argmax(axis=1),average='macro',zero_division=0,labels=np.arange(K)))
full_folds=list(get_folds(y,X=X_raw,classes=le.classes_))
seen=np.zeros(N,dtype=int)
for tr,va in full_folds:seen[va]+=1
assert len(full_folds)==5 and np.all(seen==1),'Fold 검증 실패'
rows=[]
for i,(_,va) in enumerate(full_folds,1):
    a,b=f1(y[va],p3_oof[va]),f1(y[va],p4_oof[va])
    rows.append({'fold':i,'3model_macro_f1':a,'4model_macro_f1':b,'delta':b-a})
fold_table=pd.DataFrame(rows)
display(fold_table)
full_3,full_4=f1(y,p3_oof),f1(y,p4_oof)
print(f'전체 OOF: {full_3:.9f} → {full_4:.9f} (Δ {full_4-full_3:+.9f})')
print('5개 Fold 모두 상승:',bool((fold_table.delta>0).all()))
print('팀장님 공유 0.5293 → 0.5365와 일치 여부(소수 넷째자리):',round(full_3,4)==0.5293 and round(full_4,4)==0.5365)

,fold,3model_macro_f1,4model_macro_f1,delta
0,1,0.513713,0.523521,0.009808
1,2,0.548138,0.550564,0.002426
2,3,0.513150,0.514568,0.001419
3,4,0.543897,0.557335,0.013438
4,5,0.520182,0.531627,0.011444


전체 OOF: 0.529250577 → 0.536547380 (Δ +0.007296802)
5개 Fold 모두 상승: True
팀장님 공유 0.5293 → 0.5365와 일치 여부(소수 넷째자리): True


## 3. 개발 CV·고정 홀드아웃 (선택 입력, 재학습 금지)

**주의:** 12번의 `px`, `p50`, `pg_old`, `pg_new`는 개발 4,960행에 대한 OOF이며, 12번 노트북 실행 중 메모리에만 남았을 수 있습니다. 별도 파일이 없다면 여기서 복구할 수 없습니다. **전체 OOF를 개발/홀드아웃 예측으로 위장하지 않습니다.**

선택적으로 아래 경로에 `dev_xgb.npy`, `dev_lr50.npy`, `dev_old_g3.npy`, `dev_new_g3.npy` 및 `hold_...` 4개씩 저장된 경우에만 해당 평가를 합니다. 파일이 없다면 *미검증*으로 표시합니다.

In [6]:
import importlib.util
mp=ROOT/'scripts'/'fixed_holdout_manifest.py'
spec=importlib.util.spec_from_file_location('exp13_fixed_manifest',mp)
mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
dev_idx,hold_idx=mod.verify()
assert len(dev_idx)==4960 and len(hold_idx)==1241
OPTIONAL=PRED/'exp13_inputs'

def optional_set(prefix,rows):
    paths={k:OPTIONAL/f'{prefix}_{k}.npy' for k in STEMS}
    if not all(p.exists() for p in paths.values()):
        print(prefix,'미검증: 독립 예측 파일 누락', [p.name for p in paths.values() if not p.exists()]);return None
    ans={}
    for k,p in paths.items():
        a=np.load(p,allow_pickle=False)
        if a.shape!=(rows,K) or not np.isfinite(a).all() or not np.allclose(a.sum(1),1,atol=2e-3):
            raise ValueError(f'{p}: shape 또는 확률 오류')
        ans[k]=a
    print(prefix,'예측 파일 로드 (행·클래스 순서는 생성 측과 반드시 일치해야 함)')
    return ans

dev=optional_set('dev',len(dev_idx));hold=optional_set('hold',len(hold_idx))
if dev is not None:
    d3=sum(dev[k] for k in ('xgb','lr50','old_g3'))/3
    d4=sum(dev.values())/4
    print('개발 OOF Macro-F1:',f1(y[dev_idx],d3),'→',f1(y[dev_idx],d4))
if hold is not None:
    h3=sum(hold[k] for k in ('xgb','lr50','old_g3'))/3
    h4=sum(hold.values())/4
    print('고정 홀드아웃 Macro-F1:',f1(y[hold_idx],h3),'→',f1(y[hold_idx],h4))
print('개발/홀드아웃 검증 완료 여부:',dev is not None and hold is not None)

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
dev 미검증: 독립 예측 파일 누락 ['dev_xgb.npy', 'dev_lr50.npy', 'dev_old_g3.npy', 'dev_new_g3.npy']
hold 미검증: 독립 예측 파일 누락 ['hold_xgb.npy', 'hold_lr50.npy', 'hold_old_g3.npy', 'hold_new_g3.npy']
개발/홀드아웃 검증 완료 여부: False


## 4. PR용 4모델 OOF·Test 저장

기존 파일은 덮어쓰지 않습니다. `SAVE_ENSEMBLE=True`일 때만 새 13번 결과를 기록합니다. 메타에는 입력 파일 SHA-256을 남깁니다.

In [7]:
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def save_new(path,arr):
    arr=np.asarray(arr,dtype=np.float32)
    if path.exists():
        old=np.load(path,allow_pickle=False)
        if old.shape!=arr.shape or not np.allclose(old,arr,atol=1e-7):
            raise FileExistsError(f'기존 결과와 다름, 덮어쓰기 중단: {path}')
        print('기존 동일 결과 유지:',path.name);return
    fd,tmp=tempfile.mkstemp(dir=path.parent,suffix='.npy');os.close(fd)
    try:
        np.save(tmp,arr);os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)
    print('저장:',path)

if SAVE_ENSEMBLE:
    OUT.mkdir(parents=True,exist_ok=True)
    save_new(OUT/f'{OUT_STEM}__oof.npy',p4_oof)
    save_new(OUT/f'{OUT_STEM}__test.npy',p4_test)
    meta={
      'experiment':'13_four_model_equal_ensemble','classes':CLASSES,
      'train_rows':N,'test_rows':NT,'folds':5,'weights':[0.25]*4,
      'sources':STEMS,'input_sha256':{f'{k}__{kind}':digest(PRED/f'{stem}__{kind}.npy') for k,stem in STEMS.items() for kind in ('oof','test')},
      'oof_3model_macro_f1':full_3,'oof_4model_macro_f1':full_4,
      'all_folds_improved':bool((fold_table.delta>0).all()),
      'dev_validated':dev is not None,'holdout_validated':hold is not None,
      'test_prediction_method':'mean of existing saved test probabilities; no retraining',
    }
    path=OUT/f'{OUT_STEM}__meta.json'
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8'))!=meta:raise FileExistsError('기존 메타 내용 불일치: '+str(path))
    else:path.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    display(pd.DataFrame([{'metric':'3-model OOF Macro-F1','score':full_3},{'metric':'4-model OOF Macro-F1','score':full_4}]))

저장: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds\A_model__13_four_model_equal_xgb_lr50_oldg3_newg3__oof.npy
저장: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds\A_model__13_four_model_equal_xgb_lr50_oldg3_newg3__test.npy


,metric,score
0,3-model OOF Macro-F1,0.529251
1,4-model OOF Macro-F1,0.536547


## 5. 제출 후보 (기본 비활성화)

팀장님 승인 및 독립 개발·홀드아웃 검증 이후에만 `SAVE_SUBMISSION=True`로 변경합니다. `sample_submission.csv`의 컬럼과 Test ID를 검증하고 기존 제출 파일은 덮어쓰지 않습니다.

In [8]:
if SAVE_SUBMISSION:
    if not (dev is not None and hold is not None) and not ALLOW_UNVERIFIED_SUBMISSION:
        raise RuntimeError('독립 개발/홀드아웃 예측 검증이 완료되지 않아 제출 생성 차단')
    sample_path=ROOT/'data'/'sample_submission.csv'
    if not sample_path.exists():raise FileNotFoundError(sample_path)
    sample=pd.read_csv(sample_path)
    if len(sample)!=NT:raise ValueError('sample_submission 행 수 불일치')
    if 'ID' not in sample.columns or 'SUBCLASS' not in sample.columns:
        raise ValueError('sample_submission 컬럼 구조 확인 필요')
    if 'ID' not in test.columns or not np.array_equal(sample['ID'].astype(str).to_numpy(),test['ID'].astype(str).to_numpy()):
        raise ValueError('Test ID/행 순서 불일치')
    sample['SUBCLASS']=le.inverse_transform(p4_test.argmax(axis=1))
    submission=ROOT/'submissions'/'A_model__13_four_model_equal_candidate.csv'
    submission.parent.mkdir(parents=True,exist_ok=True)
    if submission.exists():raise FileExistsError('기존 제출 파일 덮어쓰기 금지: '+str(submission))
    sample.to_csv(submission,index=False)
    print('제출 후보:',submission)
else:
    print('제출 CSV 미생성. SAVE_SUBMISSION=False (기본값)')

제출 CSV 미생성. SAVE_SUBMISSION=False (기본값)


## PR 확인

- `notebooks/A_model/13_four_model_equal_ensemble.ipynb` 및 새 `A_model__13_*` 산출물만 커밋합니다.
- 기존 10/12번 원본과 예측 파일은 변경하지 않습니다.
- PR에 **전체 OOF 개선값, Fold 5개 변화, 독립 개발 CV/홀드아웃 검증 여부**를 정확히 적습니다.
- 입력 예측 파일 누락 시 이 노트북은 의도적으로 중단합니다. 학습이 필요한 경우는 별도 작업이며, 이 실험에서 몰래 실행하지 않습니다.